In [1]:
# ============================================================
# SEARCH QUERY SPELLING CORRECTOR
# ============================================================

import nltk
import re

from nltk.corpus import words
from nltk.tokenize import word_tokenize


# ============================================================
# STEP 1: Download spelling vocabulary and tokenizer
# ============================================================

nltk.download('words')
nltk.download('punkt')
nltk.download('punkt_tab')


# ============================================================
# STEP 2: Load the spelling-error corpus
# ============================================================

# NLTK words corpus is used as the correctly spelled vocabulary
corpus_words = words.words()

# Convert all words to lowercase
vocabulary = set(word.lower() for word in corpus_words)

# Keep only alphabetic words
vocabulary = {
    word for word in vocabulary
    if word.isalpha()
}


# Add some common technical/search words
# to improve corrections for this task
vocabulary.update({
    "machine",
    "learning",
    "course",
    "courses",
    "python",
    "programming",
    "computer",
    "science",
    "artificial",
    "intelligence",
    "database",
    "software",
    "engineering",
    "technology",
    "network",
    "analysis",
    "algorithm",
    "algorithms",
    "language",
    "processing",
    "natural",
    "search",
    "query"
})


print("Spelling vocabulary loaded successfully!")
print("Vocabulary size:", len(vocabulary))


# ============================================================
# STEP 3: Calculate Levenshtein Edit Distance
# ============================================================

def edit_distance(word1, word2):

    # Create distance matrix
    rows = len(word1) + 1
    cols = len(word2) + 1

    matrix = [[0] * cols for _ in range(rows)]

    # Initialize first row
    for i in range(rows):
        matrix[i][0] = i

    # Initialize first column
    for j in range(cols):
        matrix[0][j] = j

    # Calculate edit distance
    for i in range(1, rows):
        for j in range(1, cols):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            matrix[i][j] = min(
                matrix[i - 1][j] + 1,       # Deletion
                matrix[i][j - 1] + 1,       # Insertion
                matrix[i - 1][j - 1] + cost # Substitution
            )

    return matrix[rows - 1][cols - 1]


# ============================================================
# STEP 4: Find the closest correctly spelled word
# ============================================================

def find_correction(word):

    word = word.lower()

    # If word is already correct
    if word in vocabulary:
        return word, 0

    best_word = word
    best_distance = float("inf")

    # Compare with suitable vocabulary words
    for candidate in vocabulary:

        # Ignore words that are very different in length
        if abs(len(word) - len(candidate)) > 2:
            continue

        distance = edit_distance(word, candidate)

        if distance < best_distance:
            best_distance = distance
            best_word = candidate

    return best_word, best_distance


# ============================================================
# STEP 5: Accept user search query
# ============================================================

query = input("\nEnter your search query: ")


# ============================================================
# STEP 6: Tokenize the query
# ============================================================

tokens = word_tokenize(query.lower())

# Keep only words
tokens = [
    token for token in tokens
    if re.match(r'^[a-zA-Z]+$', token)
]


# ============================================================
# STEP 7: Identify incorrect words
# ============================================================

incorrect_words = []
suggestions = {}
corrected_tokens = []


for word in tokens:

    if word in vocabulary:

        # Correct word
        corrected_tokens.append(word)

    else:

        # Incorrect word
        incorrect_words.append(word)

        correction, distance = find_correction(word)

        suggestions[word] = correction

        corrected_tokens.append(correction)


# ============================================================
# STEP 8: Display suggested corrections
# ============================================================

print("\n==============================================")
print("SEARCH QUERY SPELLING CORRECTOR")
print("==============================================")

print("\nOriginal Query:")
print(query)

print("\nIncorrect Words:")

if len(incorrect_words) == 0:
    print("No spelling errors found.")

else:
    for word in incorrect_words:
        print(word)


print("\nSuggested Corrections:")

if len(suggestions) == 0:
    print("No corrections required.")

else:
    for wrong, correct in suggestions.items():
        print(wrong, "->", correct)


# ============================================================
# STEP 9: Display corrected query
# ============================================================

corrected_query = " ".join(corrected_tokens)

print("\nFinal Corrected Query:")
print(corrected_query)


# ============================================================
# STEP 10: Test multiple spelling errors
# ============================================================

print("\n==============================================")
print("SPELLING CORRECTION COMPLETED")
print("==============================================")

[nltk_data] Downloading package words to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Spelling vocabulary loaded successfully!
Vocabulary size: 234381



Enter your search query:  machne lerning cours



SEARCH QUERY SPELLING CORRECTOR

Original Query:
machne lerning cours

Incorrect Words:
machne
lerning

Suggested Corrections:
machne -> machine
lerning -> learning

Final Corrected Query:
machine learning cours

SPELLING CORRECTION COMPLETED
